# ETL 04 - Extract: TLC Trip Record Data 2026

La comisión de Taxis y Limousinas de New York publica datos de viajes mensualmente, y están disponibles [acá](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page). 

Objetivo: encontrar los archivos Parquet publicados por NYC TLC para 2026 y descargarlos en `etl-04/data/bronze/tlc`.

En vez de adivinar todos los meses del año, leemos la página oficial y tomamos solo los links que TLC ya publicó. Esto evita confundir un `403` de CloudFront con un problema de permisos: muchas veces ese `403` simplemente significa que el archivo todavía no existe o no está publicado.


## Dependencias para toda la clase ETL 04

Antes de correr los notebooks de esta carpeta, conviene preparar el entorno con todos los paquetes que vamos a usar:

```bash
conda install -c conda-forge pandas pyarrow requests tqdm s3fs boto3 pymongo dnspython streamlit pydeck geopandas pyogrio
```

Paquetes principales:

- `pandas`, `pyarrow`: leer, inspeccionar y escribir datos Parquet.
- `requests`, `tqdm`: descargar archivos TLC y mostrar progreso.
- `s3fs`, `boto3`: subir archivos a Cloudflare R2, que usa protocolo compatible con S3.
- `pymongo`, `dnspython`: conectar con MongoDB Atlas usando URIs `mongodb+srv://...`.
- `streamlit`, `pydeck`, `geopandas`, `pyogrio`: app interactiva y mapa de zonas TLC.

Si solo vas a correr este notebook de descarga, alcanza con:

```bash
conda install -c conda-forge pandas pyarrow requests tqdm
```


In [1]:
from pathlib import Path

import time
import pandas as pd
import requests
from tqdm.auto import tqdm

PROJECT_DIR = Path.cwd()
BASE_DIR = PROJECT_DIR if PROJECT_DIR.name == "etl-04" else PROJECT_DIR / "etl-04"
BRONZE_DIR = BASE_DIR / "data" / "bronze" / "tlc"
BRONZE_DIR.mkdir(parents=True, exist_ok=True)

YEAR = 2026
TLC_PAGE_URL = "https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page"
TLC_TRIP_DATA_BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"

RECORD_TYPES = {
    "yellow": "Yellow Taxi Trip Records",
    "green": "Green Taxi Trip Records",
    "fhv": "For-Hire Vehicle Trip Records",
    "fhvhv": "High Volume For-Hire Vehicle Trip Records",
}

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/125.0 Safari/537.36"
    ),
}

# Usamos requests.get directamente para mantener el ejemplo simple.
# HEADERS simula un navegador común y ayuda a evitar bloqueos por User-Agent vacío.


## 1. Manifest desde la página oficial

Los archivos de TLC siguen un patron estable:

`{tipo}_tripdata_{año}-{mes}.parquet`

Por ejemplo: `yellow_tripdata_2026-01.parquet`.

Pero hay una trampa: si construimos URLs para meses futuros, CloudFront puede responder `403`. En este caso no significa necesariamente "no tenés permiso"; suele significar que ese objeto no existe o todavía no fue publicado.

Por eso primero descargamos la página oficial de TLC y extraemos de ahí los links reales a Parquet para 2026.


In [2]:
filename = f"yellow_tripdata_{YEAR}-01.parquet"
url = f"{TLC_TRIP_DATA_BASE_URL}/{filename}"

print(filename)
print(url)


yellow_tripdata_2026-01.parquet
https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-01.parquet


In [3]:
def fetch_tlc_page(retries: int = 3, sleep_seconds: float = 1.0) -> str:
    """Descarga la pagina oficial de TLC donde estan los links publicados."""
    last_error = None

    for attempt in range(1, retries + 1):
        try:
            response = requests.get(TLC_PAGE_URL, headers=HEADERS, timeout=30)
            response.raise_for_status()
            return response.text
        except requests.RequestException as e:
            last_error = e
            if attempt < retries:
                time.sleep(sleep_seconds * attempt)

    raise RuntimeError(f"No se pudo descargar la pagina oficial de TLC: {last_error}")


def extract_parquet_urls(html: str) -> list[str]:
    """Extrae links Parquet de TLC con una heuristica simple sobre href."""
    urls = []

    for fragmento in html.split('href="')[1:]:
        url = fragmento.split('"', 1)[0]
        es_parquet_tlc = url.startswith(TLC_TRIP_DATA_BASE_URL) and url.endswith(".parquet")
        es_tripdata_del_anio = f"_tripdata_{YEAR}-" in url

        if es_parquet_tlc and es_tripdata_del_anio:
            urls.append(url)

    return urls


def build_manifest_from_official_page() -> pd.DataFrame:
    html = fetch_tlc_page()
    records = []

    for url in extract_parquet_urls(html):
        print(f"Procesando URL: {url}")
        filename = url.rsplit("/", 1)[-1]
        record_type, year_month_ext = filename.split("_tripdata_", 1)
        year_month = year_month_ext.removesuffix(".parquet")
        year_text, month_text = year_month.split("-", 1)

        if record_type not in RECORD_TYPES or int(year_text) != YEAR:
            continue

        month = int(month_text)
        records.append(
            {
                "record_type": record_type,
                "record_label": RECORD_TYPES[record_type],
                "year": YEAR,
                "month": month,
                "year_month": year_month,
                "url": url,
                "filename": filename,
                "source": "official_tlc_page",
            }
        )

    manifest = pd.DataFrame(records).drop_duplicates()
    if manifest.empty:
        raise ValueError(f"No se encontraron links Parquet para {YEAR} en la pagina oficial.")

    return manifest.sort_values(["year_month", "record_type"]).reset_index(drop=True)



Creamos un inventario (*manifest*) con los archivos a descargar

In [4]:
manifest = build_manifest_from_official_page()

Procesando URL: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-01.parquet
Procesando URL: https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2026-01.parquet
Procesando URL: https://d37ci6vzurychx.cloudfront.net/trip-data/fhv_tripdata_2026-01.parquet
Procesando URL: https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2026-01.parquet
Procesando URL: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-02.parquet
Procesando URL: https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2026-02.parquet
Procesando URL: https://d37ci6vzurychx.cloudfront.net/trip-data/fhv_tripdata_2026-02.parquet
Procesando URL: https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2026-02.parquet
Procesando URL: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-03.parquet
Procesando URL: https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2026-03.parquet
Procesando URL: https://d37ci6vzurychx.cloudfront.n

In [5]:
manifest.head()

,record_type,record_label,year,month,year_month,url,filename,source
0,fhv,For-Hire Vehicle Trip Records,2026,1,2026-01,https://d37ci6vzurychx.cloudfront.net/trip-dat...,fhv_tripdata_2026-01.parquet,official_tlc_page
1,fhvhv,High Volume For-Hire Vehicle Trip Records,2026,1,2026-01,https://d37ci6vzurychx.cloudfront.net/trip-dat...,fhvhv_tripdata_2026-01.parquet,official_tlc_page
2,green,Green Taxi Trip Records,2026,1,2026-01,https://d37ci6vzurychx.cloudfront.net/trip-dat...,green_tripdata_2026-01.parquet,official_tlc_page
3,yellow,Yellow Taxi Trip Records,2026,1,2026-01,https://d37ci6vzurychx.cloudfront.net/trip-dat...,yellow_tripdata_2026-01.parquet,official_tlc_page
4,fhv,For-Hire Vehicle Trip Records,2026,2,2026-02,https://d37ci6vzurychx.cloudfront.net/trip-dat...,fhv_tripdata_2026-02.parquet,official_tlc_page


In [6]:
manifest = manifest.drop_duplicates().sort_values(["year_month", "record_type"])

publicados = set(zip(manifest["record_type"], manifest["year_month"]))

print("Links publicados en la pagina oficial de TLC:")
for record_type, year_month in sorted(publicados):
    print(f"  - {record_type} ({RECORD_TYPES[record_type]}), {year_month}")


Links publicados en la pagina oficial de TLC:
  - fhv (For-Hire Vehicle Trip Records), 2026-01
  - fhv (For-Hire Vehicle Trip Records), 2026-02
  - fhv (For-Hire Vehicle Trip Records), 2026-03
  - fhv (For-Hire Vehicle Trip Records), 2026-04
  - fhv (For-Hire Vehicle Trip Records), 2026-05
  - fhv (For-Hire Vehicle Trip Records), 2026-06
  - fhv (For-Hire Vehicle Trip Records), 2026-07
  - fhv (For-Hire Vehicle Trip Records), 2026-08
  - fhvhv (High Volume For-Hire Vehicle Trip Records), 2026-01
  - fhvhv (High Volume For-Hire Vehicle Trip Records), 2026-02
  - fhvhv (High Volume For-Hire Vehicle Trip Records), 2026-03
  - fhvhv (High Volume For-Hire Vehicle Trip Records), 2026-04
  - fhvhv (High Volume For-Hire Vehicle Trip Records), 2026-05
  - fhvhv (High Volume For-Hire Vehicle Trip Records), 2026-06
  - fhvhv (High Volume For-Hire Vehicle Trip Records), 2026-07
  - fhvhv (High Volume For-Hire Vehicle Trip Records), 2026-08
  - fhvhv (High Volume For-Hire Vehicle Trip Records), 202

In [7]:

manifest.to_csv(BRONZE_DIR / "tlc_2026_manifest.csv", index=False)

print(f"Archivos descargables encontrados para {YEAR}: {len(manifest)}")
display(manifest)



Archivos descargables encontrados para 2026: 35


,record_type,record_label,year,month,year_month,url,filename,source
0,fhv,For-Hire Vehicle Trip Records,2026,1,2026-01,https://d37ci6vzurychx.cloudfront.net/trip-dat...,fhv_tripdata_2026-01.parquet,official_tlc_page
1,fhvhv,High Volume For-Hire Vehicle Trip Records,2026,1,2026-01,https://d37ci6vzurychx.cloudfront.net/trip-dat...,fhvhv_tripdata_2026-01.parquet,official_tlc_page
2,green,Green Taxi Trip Records,2026,1,2026-01,https://d37ci6vzurychx.cloudfront.net/trip-dat...,green_tripdata_2026-01.parquet,official_tlc_page
3,yellow,Yellow Taxi Trip Records,2026,1,2026-01,https://d37ci6vzurychx.cloudfront.net/trip-dat...,yellow_tripdata_2026-01.parquet,official_tlc_page
4,fhv,For-Hire Vehicle Trip Records,2026,2,2026-02,https://d37ci6vzurychx.cloudfront.net/trip-dat...,fhv_tripdata_2026-02.parquet,official_tlc_page
5,fhvhv,High Volume For-Hire Vehicle Trip Records,2026,2,2026-02,https://d37ci6vzurychx.cloudfront.net/trip-dat...,fhvhv_tripdata_2026-02.parquet,official_tlc_page
6,green,Green Taxi Trip Records,2026,2,2026-02,https://d37ci6vzurychx.cloudfront.net/trip-dat...,green_tripdata_2026-02.parquet,official_tlc_page
7,yellow,Yellow Taxi Trip Records,2026,2,2026-02,https://d37ci6vzurychx.cloudfront.net/trip-dat...,yellow_tripdata_2026-02.parquet,official_tlc_page
8,fhv,For-Hire Vehicle Trip Records,2026,3,2026-03,https://d37ci6vzurychx.cloudfront.net/trip-dat...,fhv_tripdata_2026-03.parquet,official_tlc_page
9,fhvhv,High Volume For-Hire Vehicle Trip Records,2026,3,2026-03,https://d37ci6vzurychx.cloudfront.net/trip-dat...,fhvhv_tripdata_2026-03.parquet,official_tlc_page


Podemos usar `pivot_table` para ver la data de otra forma:

In [8]:
pivot = manifest.pivot_table(
    index="year_month",
    columns="record_type",
    values="url",
    aggfunc="count",
    fill_value=0,
)

display(pivot)
print("Tipos esperados:", ", ".join(RECORD_TYPES))


record_type,fhv,fhvhv,green,yellow
year_month,,,,
2026-01,1,1,1,1
2026-02,1,1,1,1
2026-03,1,1,1,1
2026-04,1,1,1,1
2026-05,1,1,1,1
2026-06,1,1,1,1
2026-07,1,1,1,1
2026-08,1,1,1,1
2026-09,0,1,1,1


Tipos esperados: yellow, green, fhv, fhvhv


## 2. Descarga

Los archivos pueden ser grandes. Si quieren practicar rapido, usen uno o dos meses en `MESES_A_DESCARGAR`. Para bajar todo lo publicado de 2026, dejen `MESES_A_DESCARGAR = None`.

In [9]:
MESES_A_DESCARGAR = None  # ejemplo para prueba: ["2026-01"]
# Quizás conviene usar la variable YEAR para filtrar directamente en el manifest.
MESES_A_DESCARGAR = ["2026-01"]

if MESES_A_DESCARGAR is None:
    manifest_descarga = manifest.copy()
else:
    manifest_descarga = manifest[manifest["year_month"].isin(MESES_A_DESCARGAR)].copy()

display(manifest_descarga[["record_type", "year_month", "filename", "url"]])


,record_type,year_month,filename,url
0,fhv,2026-01,fhv_tripdata_2026-01.parquet,https://d37ci6vzurychx.cloudfront.net/trip-dat...
1,fhvhv,2026-01,fhvhv_tripdata_2026-01.parquet,https://d37ci6vzurychx.cloudfront.net/trip-dat...
2,green,2026-01,green_tripdata_2026-01.parquet,https://d37ci6vzurychx.cloudfront.net/trip-dat...
3,yellow,2026-01,yellow_tripdata_2026-01.parquet,https://d37ci6vzurychx.cloudfront.net/trip-dat...


In [10]:
def download_file(url: str, destination: Path, retries: int = 3) -> str:
    """Descarga un archivo sin cargarlo completo en memoria."""
    chunk_size = 1024 * 1024  # 1 MB por vez
    sleep_seconds = 2.0

    if destination.exists() and destination.stat().st_size > 0:
        return "skip"

    for attempt in range(1, retries + 1):
        try:
            with requests.get(url, headers=HEADERS, stream=True, timeout=60) as response:
                if response.status_code != 200:
                    status = f"http_{response.status_code}"
                    if attempt < retries:
                        time.sleep(sleep_seconds * attempt)
                        continue
                    return status

                total = int(response.headers.get("content-length", 0))
                with destination.open("wb") as f, tqdm(
                    total=total,
                    unit="B",
                    unit_scale=True,
                    desc=destination.name,
                ) as bar:
                    for chunk in response.iter_content(chunk_size=chunk_size):
                        if chunk:
                            f.write(chunk)
                            bar.update(len(chunk))
                return "downloaded"
        except requests.RequestException as e:
            if attempt == retries:
                return f"error: {e}"
            time.sleep(sleep_seconds * attempt)



In [11]:

download_results = []
for row in manifest_descarga.itertuples(index=False):
    destination = BRONZE_DIR / row.filename
    status = download_file(row.url, destination)
    size_mb = round(destination.stat().st_size / 1024**2, 2) if destination.exists() else 0
    download_results.append(
        {
            "filename": row.filename,
            "record_type": row.record_type,
            "year_month": row.year_month,
            "status": status,
            "size_mb": size_mb,
            "path": str(destination),
        }
    )


In [12]:

downloads = pd.DataFrame(download_results)
downloads.to_csv(BRONZE_DIR / "tlc_2026_downloads.csv", index=False)
display(downloads)


,filename,record_type,year_month,status,size_mb,path
0,fhv_tripdata_2026-01.parquet,fhv,2026-01,skip,20.57,/Volumes/Data/flavioc/Codes/GitHub/cursos/Pyth...
1,fhvhv_tripdata_2026-01.parquet,fhvhv,2026-01,skip,482.43,/Volumes/Data/flavioc/Codes/GitHub/cursos/Pyth...
2,green_tripdata_2026-01.parquet,green,2026-01,skip,0.95,/Volumes/Data/flavioc/Codes/GitHub/cursos/Pyth...
3,yellow_tripdata_2026-01.parquet,yellow,2026-01,skip,61.19,/Volumes/Data/flavioc/Codes/GitHub/cursos/Pyth...


Resultado esperado de esta etapa:

- `etl-04/data/bronze/tlc/tlc_2026_manifest.csv`: inventario de archivos descargables.
- `etl-04/data/bronze/tlc/tlc_2026_manifest_expected_urls.csv`: inventario de nombres esperados y si figuran o no en la página oficial.
- `etl-04/data/bronze/tlc/*.parquet`: archivos originales descargados.
- `etl-04/data/bronze/tlc/tlc_2026_downloads.csv`: bitacora simple de descarga.

## Los archivos `.parquet`

Parquet es un formato de archivo pensado para guardar datos tabulares de forma eficiente, especialmente cuando hay muchas filas y muchas columnas.

```
4-byte magic number "PAR1"
    <Column 1 Chunk 1>
    <Column 2 Chunk 1>
    ...
    <Column N Chunk 1>
    <Column 1 Chunk 2>
    <Column 2 Chunk 2>
    ...
    <Column N Chunk 2>
    ...
    <Column 1 Chunk M>
    <Column 2 Chunk M>
    ...
    <Column N Chunk M>
    File Metadata
    4-byte length in bytes of file metadata (little endian)
    4-byte magic number "PAR1"
```


Características principales:

- Es un formato **columnar**: guarda juntos los valores de cada columna, no fila por fila como un CSV.
- Ocupa menos espacio porque suele venir **comprimido**.
- Conserva los **tipos de datos**: números, fechas, textos, booleanos, etc.
- Es rápido para análisis porque permite leer solo las columnas que necesitamos.
- Es muy usado en Data Lakes y Lakehouses porque funciona bien con grandes volúmenes de datos.

Comparado con CSV:

| Característica | CSV | Parquet |
| --- | --- | --- |
| Organización | Filas de texto | Columnas |
| Tamaño | Más grande | Más chico |
| Tipos de datos | Se infieren al leer | Se guardan en el archivo |
| Lectura parcial | Difícil | Puede leer solo algunas columnas |
| Uso típico | Intercambio simple | Analítica y datos grandes |

En este ejemplo, TLC publica los viajes en Parquet porque son archivos grandes y se consultan principalmente para análisis. Para leerlos con pandas usamos `pd.read_parquet`, que requiere tener instalado un motor como `pyarrow`.


## Nota: subir archivos grandes con Git LFS

GitHub no está pensado para versionar archivos grandes directamente con Git. Para archivos de datos pesados, como los `.parquet` descargados desde TLC, una opción es usar **Git LFS** (*Large File Storage*). Git LFS reemplaza el archivo grande en el repo por un puntero liviano, mientras el contenido real se guarda en el almacenamiento LFS asociado al repositorio.

Instalación inicial en macOS:

```bash
brew install git-lfs
git lfs install
git lfs version
```

Instalación inicial en Windows:

1. Descargar el instalador desde <https://git-lfs.com/>.
2. Ejecutar el archivo `.exe`.
3. Abrir una terminal nueva: PowerShell, Git Bash o la terminal de VSCode.
4. Activar Git LFS una vez por usuario/equipo:

```bash
git lfs install
git lfs version
```

En Windows también se puede instalar con `winget` desde PowerShell:

```powershell
winget install GitHub.GitLFS
git lfs install
git lfs version
```

Para este caso, si queremos subir los Parquet descargados en `etl-04/data/bronze/tlc`, podemos trackear ese tipo de archivo con LFS:

```bash
git lfs track "etl-04/data/bronze/tlc/*.parquet"
git add .gitattributes
git add etl-04/data/bronze/tlc/*.parquet
git commit -m "Add TLC parquet files with Git LFS"
git push
```

El archivo `.gitattributes` es importante: ahí queda registrada la regla que le dice a Git qué archivos debe manejar con LFS. Ese archivo sí se versiona normalmente.

Para verificar qué archivos están siendo manejados por LFS:

```bash
git lfs ls-files
```

En un proyecto de datos suele convenir separar dos estrategias:

- Si los datos originales son públicos y se pueden volver a descargar, muchas veces alcanza con versionar el notebook, el manifest (`tlc_2026_manifest.csv`) y la bitácora de descarga, pero dejar `etl-04/data/bronze/tlc/*.parquet` fuera de Git.
- Si queremos que el repo incluya una copia exacta de los datos usados en clase o en un análisis, entonces Git LFS es una buena opción para los `.parquet`.
